# 03. Regression Evaluation and Error Analysis

회귀 문제에서는 분류처럼 맞았다 / 틀렸다로만 끝나지 않는다.
얼마나 틀렸는지, 어디에서 크게 틀렸는지, 어떤 패턴으로 오차가 나는지를 함께 봐야 한다.

**목표**
1. 회귀 문제에서 MAE, RMSE, R²를 함께 읽는 흐름을 익힌다.
2. validation 기준으로 baseline 예측 결과를 해석한다.
3. residual(잔차)을 통해 어떤 구간에서 오차가 커지는지 확인한다.
4. error analysis 결과를 바탕으로 개선 후보를 적용해본다.
5. 최종 선택은 validation에서 하고, 마지막에만 test로 확인한다.

**주의할 점**
- 목적은 수치 결과만 보는 것이 아니라 판단 흐름을 익히는 것이다.
- 특히 R² 하나만 보고 모델을 단정하면 위험할 수 있다.
- error analysis는 단순히 그래프를 보는 것이 아니라 다음 개선 방향을 찾는 과정이다.

## 1. 왜 회귀에서는 error analysis가 중요한가

분류에서는 confusion matrix로 어디서 틀렸는지 비교적 직접적으로 볼 수 있다.

반면 회귀에서는 아래처럼 질문해야 한다.

1. 평균적으로 오차가 어느 정도인가
2. 큰 오차가 자주 나는가
3. 특정 구간에서 유독 못 맞추는가
4. 전반적으로 추세는 따라가고 있는가
5. 현재 오차 패턴을 보면 어떤 개선이 자연스러운가

즉, 회귀 평가는 단순히 `loss` 하나를 보는 것이 아니라
예측값과 실제값의 차이를 여러 관점에서 해석하는 과정이라고 볼 수 있다.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, median_absolute_error, root_mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 2. 실습 데이터

앞선 파일들과 마찬가지로 tabular 형태의 인공 데이터를 직접 생성해서 사용한다.

이번 데이터는 아래 특성을 가지도록 만든다.

- 선형 관계만 있는 것이 아니다.
- 일부 feature는 비선형 관계를 가진다.
- feature interaction이 조금 섞여 있다.
- 잡음(noise)이 완전히 균일하지 않다.
- 일부 샘플에는 큰 오차를 유발하는 noise가 섞여 있다.

즉, 아주 잘 맞는 너무 쉬운 데이터가 아니라
baseline으로 시작했을 때 개선 포인트가 어느 정도 보이는 데이터라고 보면 된다.

In [ ]:
rng = np.random.default_rng(SEED)  
n_samples = 4000  

X = pd.DataFrame({
    'feature_00': rng.normal(0, 1.0, n_samples),      # 평균 0, 표준편차 1.0인 정규분포 feature
    'feature_01': rng.normal(0, 1.2, n_samples),      # 표준편차를 조금 다르게 둔 정규분포 feature
    'feature_02': rng.uniform(-2.0, 2.0, n_samples),  # 제곱항 같은 비선형 관계를 만들기 좋은 균등분포 feature
    'feature_03': rng.uniform(-3.0, 3.0, n_samples),  # sin 함수 같은 주기성 패턴을 넣기 위한 feature
    'feature_04': rng.normal(0, 1.0, n_samples),      # interaction용 feature 1
    'feature_05': rng.normal(0, 1.0, n_samples),      # interaction용 feature 2
    'feature_06': rng.normal(0, 1.0, n_samples),      # max(., 0) 형태의 구간별 비선형 관계를 넣기 위한 feature
    'feature_07': rng.normal(0, 1.0, n_samples),      # noise 크기를 달라지게 만드는 데 사용할 feature
    'feature_08': rng.exponential(1.0, n_samples),    # 치우친(skewed) 분포를 만들기 위한 feature
    'feature_09': rng.normal(0, 1.0, n_samples),      # target 생성에는 직접 쓰지 않는 distractor feature
    'feature_10': rng.normal(0, 1.0, n_samples),      # target 생성에는 직접 쓰지 않는 distractor feature
    'feature_11': rng.normal(0, 1.0, n_samples),      # target 생성에는 직접 쓰지 않는 distractor feature
})

# noise를 완전히 일정하게 두지 않고, feature_07의 절댓값이 클수록 noise도 커지게 만든다.
# 즉, 샘플마다 오차 난이도가 조금씩 다른 이분산성(heteroscedasticity)을 일부러 넣는다.
base_noise = rng.normal(0, 1.1 + 0.9 * np.abs(X['feature_07']), n_samples)

# 전체 샘플 중 약 3%는 더 큰 noise를 추가해서, 큰 오차를 만드는 outlier성 샘플로 만든다.
# 이렇게 해야 RMSE와 MAE 차이, error analysis 필요성이 더 잘 드러난다.
outlier_mask = rng.random(n_samples) < 0.03
base_noise[outlier_mask] += rng.normal(0, 7.5, outlier_mask.sum())

y = (
    5.0 * X['feature_00']                          # 선형 관계 1: feature_00이 커질수록 target 증가
    - 3.2 * X['feature_01']                        # 선형 관계 2: feature_01이 커질수록 target 감소
    + 2.4 * (X['feature_02'] ** 2)                 # 비선형 관계: 제곱항
    + 1.9 * np.sin(1.4 * X['feature_03'])          # 비선형 관계: 사인 함수 기반 주기성 패턴
    + 2.7 * (X['feature_04'] * X['feature_05'])    # interaction: 두 feature의 곱
    + 1.5 * np.maximum(X['feature_06'], 0)         # 구간형 비선형: 0보다 클 때만 영향
    + 0.9 * np.log1p(X['feature_08'])              # 치우친 feature를 완만하게 반영하는 로그 변환 관계
    + base_noise                                   # 완벽히 맞추지 못하도록 현실적인 noise 추가
)

df = X.copy()
df['target'] = y

print('shape:', df.shape)
df.head()

In [ ]:
print(df['target'].describe().round(3))

plt.figure(figsize=(6, 3))
plt.hist(df['target'], bins=40)
plt.title('Target Distribution')
plt.xlabel('target')
plt.ylabel('count')
plt.show()

## 3. train / validation / test 분리

- train: 모델 학습
- validation: 모델 해석, 개선 방향 비교
- test: 마지막 1회 최종 확인

회귀에서도 test를 중간에 계속 보면
모델 선택과 개선 방향이 test에 맞춰질 수 있으므로 주의해야 한다.


In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,
    random_state=SEED
)

print('X_train shape:', X_train.shape)
print('X_val shape  :', X_val.shape)
print('X_test shape :', X_test.shape)

## 4. 회귀 지표를 어떻게 볼 것인가

여기서는 아래 지표를 함께 본다.

1. MAE
- 오차의 절댓값 평균
- 해석이 비교적 직관적이다.

2. RMSE
- 큰 오차에 더 민감한 지표
- outlier나 큰 miss를 더 강하게 반영한다.

3. R²
- 분산을 얼마나 설명하는가를 보는 지표
- 1에 가까울수록 좋지만, 이것만으로 충분하지는 않다.

4. Median Absolute Error
- 중앙값 기준 절대 오차
- 일부 큰 오차 때문에 평균이 흔들릴 때 함께 참고하기 좋다.

즉,
MAE는 전반적 평균 오차,
RMSE는 큰 오차 민감도,
R²는 설명력,
Median AE는 typical한 오차 수준을 보는 느낌으로 읽으면 된다.


In [ ]:
def evaluate_regressor(model, X, y_true):
    pred = model.predict(X) 

    metrics = {
        'mae': mean_absolute_error(y_true, pred),              # 절대 오차의 평균. 평균적으로 얼마나 틀렸는지 직관적으로 보기 좋다.
        'rmse': root_mean_squared_error(y_true, pred),         # 큰 오차에 더 민감한 지표. outlier나 큰 miss의 영향을 더 크게 반영한다.
        'r2': r2_score(y_true, pred),                          # target의 분산을 얼마나 설명하는지 보는 지표. 전체 추세를 얼마나 따라가는지 확인할 때 본다.
        'median_abs_error': median_absolute_error(y_true, pred),  # 절대 오차의 중앙값. 일부 큰 오차에 덜 흔들리는 typical error 수준을 보기 좋다.
    }
    return metrics, pred  


def make_error_frame(y_true, y_pred):
    error_df = pd.DataFrame({
        'y_true': y_true, 
        'y_pred': y_pred,  
    })
    error_df['residual'] = error_df['y_true'] - error_df['y_pred']   # 잔차(residual): 실제값 - 예측값. 양수면 과소예측, 음수면 과대예측이다.
    error_df['abs_error'] = error_df['residual'].abs()               # 절대 오차: 오차 크기만 본다. 방향보다 얼마나 크게 틀렸는지 볼 때 사용한다.
    error_df['squared_error'] = error_df['residual'] ** 2            # 제곱 오차: 큰 오차를 더 크게 반영한다. RMSE 같은 지표 계산에 활용하기 좋다.
    return error_df


def summarize_errors(error_df):
    return pd.DataFrame([{
        'residual_mean': error_df['residual'].mean(),                 # 잔차 평균: 0에 가까우면 전체적으로 한쪽으로 치우친 예측이 덜하다고 볼 수 있다.
        'residual_std': error_df['residual'].std(),                   # 잔차 표준편차: 잔차의 퍼짐 정도. 오차가 얼마나 들쭉날쭉한지 본다.
        'mae': error_df['abs_error'].mean(),                          # 절대 오차 평균: 샘플 전체에서 평균적으로 어느 정도 틀리는지 다시 요약한다.
        'median_abs_error': error_df['abs_error'].median(),           # 절대 오차 중앙값: 일부 큰 오차의 영향보다 전형적인 오차 수준을 보는 데 유리하다.
        'p90_abs_error': error_df['abs_error'].quantile(0.90),        # 절대 오차 90퍼센타일: 상위 큰 오차 구간이 어느 정도인지 본다.
        'under_prediction_ratio': (error_df['residual'] > 0).mean(),  # 과소예측 비율: 실제값 > 예측값인 비율. 모델이 낮게 예측하는 경향이 얼마나 있는지 본다.
        'over_prediction_ratio': (error_df['residual'] < 0).mean(),   # 과대예측 비율: 실제값 < 예측값인 비율. 모델이 높게 예측하는 경향이 얼마나 있는지 본다.
    }])

## 5. baseline 모델

앞서서 baseline 후보를 비교하는 흐름을 다루었으므로, 여기서는 그 중 하나를 골라 더 깊게 해석하도록 한다.

선형 baseline으로 `Ridge`를 사용한다.

이 모델은 다음 특징이 있다.

- 해석이 비교적 단순하다.
- 스케일링된 선형 관계를 잘 잡는다.
- 비선형 관계나 interaction이 강하면 한계가 드러날 수 있다.

즉, 이번 데이터에서는 일부를 설명하되
error analysis를 통해 부족한 부분이 보일 가능성이 있다.

In [ ]:
# TODO: 회귀 baseline 모델을 만들어서 validation 데이터에서 평가
baseline_model = Pipeline([])

baseline_model.fit(X_train, y_train)
val_metrics, val_pred = evaluate_regressor(baseline_model, X_val, y_val)

print('validation metrics')
for k, v in val_metrics.items():
    print(f'- {k}: {v:.4f}')


## 6. baseline validation 결과를 어떻게 읽을까

숫자를 볼 때는 아래처럼 같이 생각하는 편이 좋다.

- MAE가 너무 크면 평균적으로도 잘 못 맞추는 것이다.
- RMSE가 MAE보다 상대적으로 더 크게 느껴지면 큰 오차 샘플이 섞여 있을 수 있다.
- R²가 아주 낮지는 않다면 전체 추세는 어느 정도 따라가고 있을 수 있다.
- 하지만 R²가 괜찮아 보여도 특정 구간에서 체계적으로 틀릴 수 있다.

그래서 여기서 끝내지 않고
반드시 예측값과 residual을 함께 봐야 한다.


In [ ]:
# TODO: validation 데이터에서 예측값과 실제값의 차이를 분석
val_error_df = make_error_frame(____, ____)  

val_error_summary = summarize_errors(____)  
val_error_summary.round(4)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val, val_pred, alpha=0.35)
min_v = min(y_val.min(), val_pred.min())
max_v = max(y_val.max(), val_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')
plt.title('Validation: Actual vs Predicted (Baseline)')
plt.xlabel('y_true')
plt.ylabel('y_pred')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(val_error_df['residual'], bins=40)
plt.title('Validation Residual Distribution (Baseline)')
plt.xlabel('residual = y_true - y_pred')
plt.ylabel('count')
plt.show()

plt.figure(figsize=(6, 4))
plt.scatter(val_pred, val_error_df['residual'], alpha=0.35)
plt.axhline(0, linestyle='--')
plt.title('Validation Residual vs Predicted (Baseline)')
plt.xlabel('y_pred')
plt.ylabel('residual')
plt.show()

## 7. residual을 볼 때 확인할 점

1. residual 평균이 0 근처인가
- 크게 한쪽으로 치우치면 전반적 bias를 의심할 수 있다.

2. 특정 예측 구간에서 residual 폭이 넓어지는가
- 구간별 난이도 차이나 이분산성을 의심할 수 있다.

3. actual vs predicted에서 직선 대각선에서 체계적으로 벗어나는가
- 선형 모델이 못 잡는 비선형 패턴이 있을 수 있다.

4. 아주 큰 오차 샘플이 일부 존재하는가
- outlier, noise, 드문 패턴을 의심할 수 있다.


In [ ]:
# 오차가 큰 샘플 상위 10개 확인
# TODO: 큰 오차 기준 컬럼 작성
val_error_df_sorted = val_error_df.sort_values(____, ascending=False).reset_index(drop=True)  
val_error_df_sorted.head(10).round(4)

In [ ]:
# target 구간별 오차 비교
# TODO: 몇 개 구간으로 나눌지 입력
target_bin_df = val_error_df.copy()
target_bin_df['target_bin'] = pd.qcut(target_bin_df['y_true'], q=____, duplicates='drop')  

# TODO: target_bin별로 count, mean_true, mae, rmse, mean_residual 계산해서 요약 테이블 만들기
target_bin_summary = (
    target_bin_df
    .groupby('target_bin', observed=False)
    .agg(
        count=('y_true', 'size'),
        mean_true=('y_true', 'mean'),
        mae=('abs_error', '____'),  
        rmse=('squared_error', lambda s: np.sqrt(____)),  
        mean_residual=('residual', '____') 
    )
    .reset_index()
)

target_bin_summary.round(4)


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(range(len(target_bin_summary)), target_bin_summary['mae'])
plt.title('Validation MAE by Target Bin (Baseline)')
plt.xlabel('target bin')
plt.ylabel('mae')
plt.xticks(range(len(target_bin_summary)), [f'bin_{i+1}' for i in range(len(target_bin_summary))])
plt.show()

## 8. baseline error analysis 해석

이 단계에서 자주 나오는 해석은 아래와 같다.

1. 전체 추세는 어느 정도 따라가지만 큰 오차 샘플이 남아 있다.
2. 특정 target 구간에서 MAE가 더 크면 구간별 난이도 차이가 있을 수 있다.
3. residual plot에서 패턴이 보이면 선형 모델이 놓치는 비선형 관계를 의심할 수 있다.
4. RMSE가 MAE보다 꽤 크면 일부 큰 miss가 전체 성능을 끌어내리고 있을 수 있다.

이번 데이터는 생성할 때부터 비선형 관계와 interaction을 조금 섞어두었기 때문에 선형 baseline 하나로는 한계가 드러나는 흐름을 기대할 수 있다.

## 9. 개선 후보 적용

error analysis 결과를 보고 이번에는 비선형 관계와 interaction을 더 잘 잡을 수 있는 모델을 하나 비교해본다.

여기서는 `RandomForestRegressor`를 사용한다.

이 모델은 다음 특징이 있다.

- 선형식으로 직접 표현하지 않아도 된다.
- 비선형 분할을 통해 복잡한 관계를 어느 정도 잡을 수 있다.
- 다만 완전히 자동으로 모든 문제가 해결되는 것은 아니고,
  여전히 noise와 outlier 영향은 남을 수 있다.

즉,
이번 개선의 목적은 완벽한 모델 만들기가 아니라
error analysis를 바탕으로 합리적인 다음 후보를 적용해보는 것이다.

In [ ]:
improved_model = RandomForestRegressor(
    n_estimators=220,
    max_depth=12,
    min_samples_leaf=3,
    random_state=SEED,
    n_jobs=-1
)

improved_model.fit(X_train, y_train)
improved_val_metrics, improved_val_pred = evaluate_regressor(improved_model, X_val, y_val)

print('improved validation metrics')
for k, v in improved_val_metrics.items():
    print(f'- {k}: {v:.4f}')


In [ ]:
# TODO: improved 모델의 validation 데이터에서 예측값과 실제값의 차이를 분석
comparison_df = pd.DataFrame([
    {'model': 'ridge_baseline', **______},           
    {'model': 'random_forest_improved', **______},
])

comparison_df.round(4)


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_val, improved_val_pred, alpha=0.35)
min_v = min(y_val.min(), improved_val_pred.min())
max_v = max(y_val.max(), improved_val_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')
plt.title('Validation: Actual vs Predicted (Improved)')
plt.xlabel('y_true')
plt.ylabel('y_pred')
plt.show()

improved_val_error_df = make_error_frame(y_val, improved_val_pred)

plt.figure(figsize=(6, 4))
plt.scatter(improved_val_pred, improved_val_error_df['residual'], alpha=0.35)
plt.axhline(0, linestyle='--')
plt.title('Validation Residual vs Predicted (Improved)')
plt.xlabel('y_pred')
plt.ylabel('residual')
plt.show()

## 10. 어떤 모델을 최종 선택할 것인가

최종 선택은 test가 아니라 validation 기준으로 한다.

- MAE가 더 낮은가
- RMSE도 같이 낮아졌는가
- R²가 개선되었는가
- residual 패턴이 조금 더 완화되었는가

실제 프로젝트에서는
비즈니스 해석 가능성, 속도, 운영 제약까지 함께 고려할 수도 있다.

In [ ]:
# TODO: 최종 선택에 사용할 지표 이름 채워서, baseline과 improved 모델 중 최종 선택
if improved_val_metrics['____'] < val_metrics['____']:  
    final_model_name = 'random_forest_improved'
    final_model = improved_model
else:
    final_model_name = 'ridge_baseline'
    final_model = baseline_model

print('최종 선택 모델:', final_model_name)

## 11. test는 마지막에만 확인

최종 선택 모델을 test에 적용한다.

중요한 점은 test 결과를 본 뒤 다시 모델을 바꾸거나 튜닝하면 안 된다는 것이다.

그렇게 되면 test가 최종 평가가 아니라 또 다른 validation처럼 변해버린다.

In [ ]:
test_metrics, test_pred = evaluate_regressor(final_model, X_test, y_test)

print('test metrics')
for k, v in test_metrics.items():
    print(f'- {k}: {v:.4f}')

In [ ]:
test_error_df = make_error_frame(y_test, test_pred)

test_result_df = pd.DataFrame([{
    'selected_model': final_model_name,
    **test_metrics
}])

test_result_df.round(4)

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.35)
min_v = min(y_test.min(), test_pred.min())
max_v = max(y_test.max(), test_pred.max())
plt.plot([min_v, max_v], [min_v, max_v], linestyle='--')
plt.title('Test: Actual vs Predicted')
plt.xlabel('y_true')
plt.ylabel('y_pred')
plt.show()

plt.figure(figsize=(6, 4))
plt.hist(test_error_df['residual'], bins=40)
plt.title('Test Residual Distribution')
plt.xlabel('residual')
plt.ylabel('count')
plt.show()

## 12. validation과 test를 함께 읽을 때 주의할 점

validation에서 좋아진 방향이
test에서도 대체로 유지되면 좋은 흐름이다.

다만 숫자가 완전히 똑같이 나올 필요는 없다.

- validation도 표본이다.
- test도 또 다른 표본이다.
- noise와 outlier가 있으면 split마다 조금씩 차이가 난다.

따라서 해석은 아래처럼 하는 편이 좋다.

1. validation에서 개선 방향을 정한다.
2. test에서 그 방향이 대체로 유지되는지 본다.
3. 약간의 차이는 자연스럽게 받아들인다.
4. 큰 차이가 나면 과적합, 분할 편차, 데이터 난이도 차이를 의심한다.


In [ ]:
# TODO: 최종 선택한 모델의 validation과 test 지표를 한 테이블에 요약해서 보기 좋게 출력
final_val_metrics, _ = evaluate_regressor(final_model, X_val, y_val)

summary_df = pd.DataFrame([
    {'split': 'validation', **______},  
    {'split': 'test', **______},        
])

summary_df.round(4)

## 정리

1. 회귀에서는 MAE, RMSE, R²를 함께 봐야 한다.
2. RMSE가 MAE보다 더 민감하게 움직이면 큰 오차 샘플의 영향을 의심할 수 있다.
3. actual vs predicted, residual plot, 큰 오차 샘플 확인은 error analysis의 핵심이다.
4. error analysis는 끝이 아니라 다음 개선 방향을 찾는 과정이다.
5. 개선 후보 비교와 최종 선택은 validation에서 하고, 마지막에만 test를 확인해야 한다.

즉,
회귀 평가의 핵심은
오차를 읽고 다음 판단으로 연결하는 것이다.

## 오차 패턴별 해석과 다음 개선 방향

| 패턴                                   | 해석                              | 개선 방향                                         |
| ------------------------------------ | ------------------------------- | --------------------------------------------- |
| MAE는 보통인데 RMSE가 큼                    | 일부 샘플에서 매우 크게 틀리고 있을 가능성이 크다    | 큰 오차 샘플 확인, outlier·데이터 오류 점검, 비선형 모델 비교      |
| residual_mean이 0에서 멂                 | 전체적으로 과소예측 또는 과대예측으로 치우쳐 있다     | 중요한 feature 누락 여부 확인, 더 유연한 모델 비교             |
| 과소예측 또는 과대예측 비율이 한쪽으로 치우침            | 특정 구간을 충분히 따라가지 못하고 있다          | 큰 값/작은 값 구간의 오차를 따로 보고 feature나 모델 보강 검토      |
| 낮은 target 구간 과대예측, 높은 target 구간 과소예측 | 극단값을 못 따라가고 예측이 가운데로 모이는 경향이 있다 | 트리 계열 모델 비교, 제곱항·interaction·변환 검토            |
| residual plot에 곡선 패턴이 보임             | 선형 모델이 설명하지 못한 비선형 관계가 남아 있다    | polynomial feature, log 변환, 트리/boosting 계열 검토 |
| 특정 구간으로 갈수록 residual 퍼짐이 커짐          | 이분산성 가능성이 있다                    | 구간별 오차 요약, target 변환, 해당 구간 feature 보강 검토     |
| 큰 오차 샘플이 소수 존재                       | 평균 성능은 괜찮아도 일부 케이스에서 위험할 수 있다   | 상위 오차 샘플 직접 확인, 공통 패턴·입력 오류·이상치 점검            |
| train은 좋은데 validation/test가 나쁨       | 과적합 또는 일반화 실패 가능성이 있다           | 모델 단순화, 규제 강화, 교차검증, 데이터 분할/누수 점검             |
